## MCMC plotting

This notebook loads a chain file stored in netCDF format as an `arviz.InferenceData` (`xarray.DataTree`) object. We make several diagnostic plots for the MCMC with `arviz_plots`, and a corner plot with `corner.corner`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import corner
import xarray as xr
from arviz_base.labels import MapLabeller
import arviz_plots as azp

%config InlineBackend.figure_format = "retina"
azp.style.use("arviz-variat")

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    """Walk parents for `pyproject.toml` so defaults work from any cwd inside the repo."""
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").is_file():
            return parent
    return p


def load_inference_datatree(path: Path) -> xr.DataTree:
    """Open InferenceData-on-disk as an xarray DataTree (matches `azb.from_numpyro` round-trip)."""
    path = path.expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    suffix = path.suffix.lower()
    if suffix in {".h5", ".hdf5"}:
        return xr.open_datatree(path, engine="h5netcdf")
    return xr.open_datatree(path)


def infer_scalar_var_names(dt: xr.DataTree, *, group: str = "posterior") -> list[str]:
    """Variables with only `(chain, draw)` dimensions — same role as `settings.active_params` in the analysis notebook."""
    ds = dt[group]
    out: list[str] = []
    for name in ds.data_vars:
        dims = set(ds[name].dims)
        if dims == {"chain", "draw"}:
            out.append(name)
    return sorted(out)

In [ ]:
# Copied from `notebooks/analysis_numpyro.ipynb` (fiducial hyperparameters / cosmology).
FIDUCIALS: dict[str, float] = {
    "H0": 67.66,
    "Omega_m": 0.3096,
    "chi0": 1.0,
    "chin": 1.91,
    "gamma": 2.7,
    "kappa": 2.9,
    "z_peak": 1.9,
}
VAR_LABELS = {
    "H0": r"$H_0$",
    "Omega_m": r"$\Omega_m$",
    "chi0": r"$\chi_0$",
    "chin": r"$\chi_n$",
    "gamma": r"$\gamma$",
    "kappa": r"$\kappa$",
    "z_peak": r"$z_\mathrm{peak}$",
}
labeller = MapLabeller(var_name_map=VAR_LABELS)
# FIDUCIALS["omega_m"] = FIDUCIALS["Omega_m"] * (FIDUCIALS["H0"] / 100.0) ** 2

# Default: sample file produced by `analysis_numpyro.ipynb` when `settings.outdir` / `settings.label` match.
INFERENCE_DATA_PATH = (
    find_repo_root()
    / "chains/mcmc-H0-Omega_m-det=S1,R1-seed42-20260630-013127.nc"
)

# Set to a non-empty list to override automatic detection (e.g. only cosmology parameters).
VAR_NAMES: list[str] | None = None
PLOT_EXTRA_FIELDS: bool = False

inference_data = load_inference_datatree(INFERENCE_DATA_PATH)
var_names = list(VAR_NAMES) if VAR_NAMES else infer_scalar_var_names(inference_data)
if not PLOT_EXTRA_FIELDS:
    var_names = [n for n in var_names if n in FIDUCIALS]
var_names

In [ ]:
# inference_data
inference_data

### Corner plot

In [ ]:
corner.corner(
    inference_data,
    var_names=var_names,
    labeller=labeller,
    divergences=True,
    truths={k: FIDUCIALS[k] for k in var_names if k in FIDUCIALS},
    truth_color="C3",
    quantiles=[0.16, 0.5, 0.84],
)

### MCMC diagnostics

Plots: trace, autocorrelation, convergence, ESS, ESS evolution, rank.

In [ ]:
azp.plot_trace_dist(inference_data, var_names=var_names, labeller=labeller)

In [ ]:
azp.plot_autocorr(inference_data, var_names=var_names, max_lag=300, labeller=labeller)

In [ ]:
azp.plot_convergence_dist(inference_data, var_names=var_names, ref_line=True, labeller=labeller)

In [ ]:
azp.plot_ess(inference_data, var_names=var_names, extra_methods=True, labeller=labeller)

In [ ]:
azp.plot_ess_evolution(inference_data, var_names=var_names, extra_methods=True, labeller=labeller)

In [ ]:
azp.plot_rank(inference_data, var_names=var_names, labeller=labeller)